# Bolt 2 Interactive Lab: Real Data Sanity Check
This notebook verifies the Data Pipeline Core using the actual Store Sales competition data.


In [4]:

import sys
import os
from pathlib import Path

# Add project root to path so we can import 'src'
root_path = Path(os.getcwd()).parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

print(f"Project root added to path: {root_path}")

from src.data.loader import DataLoader
from pathlib import Path
import pandas as pd

# Load competition data
data_dir = Path('../data/raw')
loader = DataLoader()

# Load main training data
print("--- Loading Train Data ---")
train_df = loader.load(data_dir / 'train.csv')
display(train_df.head())
print(f"Train Dtypes:\n{train_df.dtypes}\n")

# Load ancillary data (Oil prices)
print("--- Loading Oil Data ---")
oil_df = loader.load(data_dir / 'oil.csv')
display(oil_df.head())
print(f"Oil Dtypes:\n{oil_df.dtypes}\n")


Project root added to path: /mnt/mac/Users/rauldemaio/Projects Local/kaggle-store-sales-ts
--- Loading Train Data ---


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


Train Dtypes:
id                      int64
date           datetime64[us]
store_nbr               int64
family                    str
sales                 float64
onpromotion             int64
dtype: object

--- Loading Oil Data ---


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


Oil Dtypes:
date          datetime64[us]
dcoilwtico           float64
dtype: object



In [5]:

from src.pipeline.factory import FeaturePipelineFactory
import pandas as pd
import numpy as np

# Let's test the imputer on the real Oil data (which has missing values)
print("--- Testing YAML Pipeline on Real Oil Data ---")

yaml_conf = """
steps:
  - name: "oil_imputer"
    class: "src.features.impute.TimeSeriesImputer"
    params:
      method: "interpolate"
      columns: ["dcoilwtico"]
"""

# Reload oil data to be fresh
oil_df = loader.load(data_dir / 'oil.csv')
print(f"Missing values before: {oil_df['dcoilwtico'].isna().sum()}")

factory = FeaturePipelineFactory()
pipe = factory.create_from_yaml(yaml_conf)
oil_processed = pipe.fit_transform(oil_df)

print(f"Missing values after:  {oil_processed['dcoilwtico'].isna().sum()}")
display(oil_processed.head(10))


--- Testing YAML Pipeline on Real Oil Data ---
Missing values before: 43
Missing values after:  1


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
5,2013-01-08,93.21
6,2013-01-09,93.08
7,2013-01-10,93.81
8,2013-01-11,93.60
9,2013-01-14,94.27
